In [ ]:
# ==============================================================================
# GERADOR DE DASHBOARD HTML ESTÁTICO (VERSÃO FINAL - LEGENDA CORRIGIDA)
# ==============================================================================
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive', force_remount=False)

# 2. Conectar nos bancos
base_path = "/content/drive/MyDrive"
bancos = {
    "ETL1": os.path.join(base_path, "Backup_ETL1_Recife/mobilidade_recife.db"),
    "ETL2": os.path.join(base_path, "Backup_ETL2_Recife/infracoes_recife.db"),
    "ETL3": os.path.join(base_path, "Backup_ETL3_Recife/ciclovias_recife.db"),
    "ETL4": os.path.join(base_path, "Backup_ETL4_Recife/estacoes_recife.db"),
    "ETL5": os.path.join(base_path, "Backup_ETL5_Recife/dados/tratados/etl5_equipamentos.db"),
}

dados = {}
for nome, path in bancos.items():
    if os.path.exists(path):
        con = sqlite3.connect(path)
        if nome == "ETL1":
            dados[nome] = {"fato": pd.read_sql("SELECT * FROM fato_chamados", con)}
        elif nome == "ETL2":
            dados[nome] = {"fato": pd.read_sql("SELECT * FROM fato_infracoes", con)}
        elif nome == "ETL3":
            dados[nome] = {
                "fato": pd.read_sql("SELECT * FROM fato_segmentos_cicloviarios", con),
                "dim_bairro": pd.read_sql("SELECT * FROM dim_bairro", con),
            }
        elif nome == "ETL4":
            dados[nome] = {"fato": pd.read_sql("SELECT * FROM fato_estacoes", con)}
        elif nome == "ETL5":
            dados[nome] = {
                "fato": pd.read_sql("SELECT * FROM dim_equipamento", con),
                "agg_bairro": pd.read_sql("SELECT * FROM agg_equip_bairro", con),
            }
        con.close()
        print(f"✅ {nome}: {len(dados[nome]['fato'])} registros carregados")

print("\n🎨 Gerando gráficos com legenda corrigida...")

# --- ETL 1: Chamados ---
df_chamados = dados.get("ETL1", {}).get("fato", pd.DataFrame())
grafico_serie_mensal = grafico_matriz_calor = grafico_top_bairros_chamados = None
if not df_chamados.empty:
    if 'mes' in df_chamados.columns or 'nome_mes' in df_chamados.columns:
        col_mes = 'nome_mes' if 'nome_mes' in df_chamados.columns else 'mes'
        ordem_meses = ['01 - Janeiro', '02 - Fevereiro', '03 - Março', '04 - Abril', '05 - Maio', '06 - Junho',
                      '07 - Julho', '08 - Agosto', '09 - Setembro', '10 - Outubro', '11 - Novembro', '12 - Dezembro']
        df_chamados[col_mes] = pd.Categorical(df_chamados[col_mes], categories=ordem_meses, ordered=True)
        serie_mensal = df_chamados.groupby(col_mes, observed=False).size().reset_index(name='qtd')
        grafico_serie_mensal = px.line(serie_mensal, x=col_mes, y='qtd', title='Evolução Mensal dos Chamados (2024)', markers=True, color_discrete_sequence=['#1f77b4'])

    if 'dia_semana' in df_chamados.columns and 'faixa_horaria' in df_chamados.columns:
        matriz = df_chamados.groupby(['dia_semana', 'faixa_horaria']).size().reset_index(name='qtd')
        pivot = matriz.pivot(index='dia_semana', columns='faixa_horaria', values='qtd').fillna(0)
        grafico_matriz_calor = px.imshow(pivot, labels=dict(x="Faixa Horária", y="Dia da Semana", color="Qtd."), color_continuous_scale='Reds', title='Concentração de Chamados por Dia e Hora')

    if 'bairro' in df_chamados.columns:
        top_bairros = df_chamados['bairro'].value_counts().head(10).reset_index()
        top_bairros.columns = ['bairro', 'qtd']
        grafico_top_bairros_chamados = px.bar(top_bairros, x='bairro', y='qtd', title='Top 10 Bairros com Mais Chamados', color='qtd', color_continuous_scale='Blues')

# --- ETL 2: Infrações ---
df_infracoes = dados.get("ETL2", {}).get("fato", pd.DataFrame())
grafico_top_infracoes = None
if not df_infracoes.empty and 'descricao_infracao' in df_infracoes.columns:
    top_tipos = df_infracoes['descricao_infracao'].value_counts().head(15).reset_index()
    top_tipos.columns = ['descricao_infracao', 'qtd']
    grafico_top_infracoes = px.bar(top_tipos, x='qtd', y='descricao_infracao', orientation='h',
                                  title='Top 15 Tipos de Infração Mais Registrados',
                                  color='qtd', color_continuous_scale='Oranges')
    grafico_top_infracoes.update_layout(yaxis={'categoryorder': 'total ascending', 'tickfont': {'size': 11}})

# --- ETL 3: Malha Cicloviária ---
df_ciclovias = dados.get("ETL3", {}).get("fato", pd.DataFrame())
grafico_pie_categoria = grafico_densidade = None
if not df_ciclovias.empty:
    if 'Categoria' in df_ciclovias.columns:
        por_categoria = df_ciclovias.groupby('Categoria')['comprimento_km'].sum().reset_index()
        grafico_pie_categoria = px.pie(por_categoria, values='comprimento_km', names='Categoria',
                                      title='Distribuição por Categoria',
                                      color_discrete_map={'Permanente': '#2ca02c', 'Operacional': '#ff7f0e'})

    dim_bairro = dados.get("ETL3", {}).get("dim_bairro")
    if dim_bairro is not None and 'area_km2' in dim_bairro.columns and 'id_bairro' in df_ciclovias.columns:
        densidade = df_ciclovias.groupby('id_bairro')['comprimento_km'].sum().reset_index()
        densidade = densidade.merge(dim_bairro, on='id_bairro', how='left')
        densidade['densidade'] = densidade['comprimento_km'] / densidade['area_km2'].replace(0, np.nan)
        densidade = densidade.sort_values('densidade', ascending=False).head(15)
        grafico_densidade = px.bar(densidade, x='Nome_Bairro_Oficial', y='densidade',
                                  title='Top 15 Bairros por Densidade (km/km²)',
                                  color='densidade', color_continuous_scale='Greens')

# --- ETL 4: Estações ---
df_estacoes = dados.get("ETL4", {}).get("fato", pd.DataFrame())
grafico_estacoes_bairro = grafico_proximidade = None
if not df_estacoes.empty:
    bairro_col = 'Nome_Bairro_Oficial' if 'Nome_Bairro_Oficial' in df_estacoes.columns else 'bairro'
    if bairro_col in df_estacoes.columns:
        por_bairro = df_estacoes.groupby(bairro_col).size().reset_index(name='qtd')
        por_bairro = por_bairro.sort_values('qtd', ascending=False).head(15)
        grafico_estacoes_bairro = px.bar(por_bairro, x=bairro_col, y='qtd',
                                        title='Top 15 Bairros com Mais Estações',
                                        color='qtd', color_continuous_scale='Blues')

    if 'distancia_m' in df_estacoes.columns:
        grafico_proximidade = px.histogram(df_estacoes, x='distancia_m', nbins=30,
                                          title='Distribuição das Distâncias até a Malha Permanente',
                                          color_discrete_sequence=['#1f77b4'])
        grafico_proximidade.add_vline(x=300, line_dash="dash", line_color="red", annotation_text="Limite 300m")

# --- ETL 5: Equipamentos + Dispersão (CORRIGIDO) ---
df_equip = dados.get("ETL5", {}).get("fato", pd.DataFrame())
grafico_equip_bairro = grafico_dispersao = None
if not df_equip.empty:
    agg_bairro = dados.get("ETL5", {}).get("agg_bairro")
    if agg_bairro is not None and not agg_bairro.empty:
        grafico_equip_bairro = px.bar(agg_bairro.sort_values('quantidade_equipamentos', ascending=False).head(15),
                                     x='bairro', y='quantidade_equipamentos',
                                     title='Top 15 Bairros com Mais Equipamentos',
                                     color='quantidade_equipamentos', color_continuous_scale='Reds')

    if 'bairro' in df_equip.columns and 'ETL1' in dados:
        df_chamados_agg = dados['ETL1']['fato'].groupby('bairro').size().reset_index(name='qtd_chamados')
        df_equip_agg = df_equip.groupby('bairro').size().reset_index(name='qtd_equipamentos')
        df_merged = df_equip_agg.merge(df_chamados_agg, on='bairro', how='left')
        df_merged['qtd_chamados'] = df_merged['qtd_chamados'].fillna(0).astype(int)

        grafico_dispersao = px.scatter(df_merged, x='qtd_equipamentos', y='qtd_chamados',
                                      hover_name='bairro',
                                      title='Equipamentos × Chamados (Associação Descritiva)',
                                      color='qtd_equipamentos', color_continuous_scale='Viridis',
                                      size='qtd_chamados', size_max=20)

        # CORREÇÃO: showlegend=False para não sobrepor a legenda de cor
        if len(df_merged) > 2:
            z = np.polyfit(df_merged['qtd_equipamentos'], df_merged['qtd_chamados'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(df_merged['qtd_equipamentos'].min(), df_merged['qtd_equipamentos'].max(), 100)
            grafico_dispersao.add_trace(go.Scatter(x=x_line, y=p(x_line), mode='lines',
                                                  name='Tendência',
                                                  line=dict(color='red', dash='dash'),
                                                  showlegend=False))

print("✅ Gráficos gerados com legenda corrigida!")

# 3. Montar HTML (mesmo código anterior)
html_content = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard Mobilidade Urbana Recife 2024</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Segoe UI', Tahoma, sans-serif; background: #f5f5f5; }}
        .header {{ background: linear-gradient(135deg, #1f77b4, #2ca02c); color: white; padding: 30px; text-align: center; }}
        .header h1 {{ font-size: 2em; margin-bottom: 10px; }}
        .header p {{ opacity: 0.9; }}
        .kpi-container {{ display: flex; flex-wrap: wrap; gap: 20px; padding: 20px; justify-content: center; background: white; }}
        .kpi {{ background: #f8f9fa; border-left: 4px solid #1f77b4; padding: 15px 25px; border-radius: 5px; min-width: 180px; text-align: center; }}
        .kpi h3 {{ font-size: 0.9em; color: #666; margin-bottom: 5px; }}
        .kpi p {{ font-size: 1.8em; font-weight: bold; color: #1f77b4; }}
        .tabs {{ display: flex; background: white; border-bottom: 2px solid #ddd; padding: 0 20px; flex-wrap: wrap; }}
        .tab {{ padding: 15px 25px; cursor: pointer; border: none; background: none; font-size: 1em; font-weight: 500; color: #666; }}
        .tab.active {{ color: #1f77b4; border-bottom: 3px solid #1f77b4; }}
        .tab-content {{ display: none; padding: 30px; background: white; margin: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .tab-content.active {{ display: block; }}
        .chart-container {{ margin: 30px 0; }}
        .warning {{ background: #fff3cd; border-left: 4px solid #ffc107; padding: 15px; margin: 20px; border-radius: 5px; }}
        .footer {{ text-align: center; padding: 20px; color: #666; font-size: 0.9em; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🚦 Dashboard de Mobilidade Urbana — Recife 2024</h1>
        <p>5 ETLs Integrados | Fonte: Portal de Dados Abertos do Recife</p>
    </div>

    <div class="kpi-container">
        <div class="kpi"><h3>Chamados (ETL1)</h3><p>{len(df_chamados):,}</p></div>
        <div class="kpi"><h3>Infrações (ETL2)</h3><p>{len(df_infracoes):,}</p></div>
        <div class="kpi"><h3>Segmentos (ETL3)</h3><p>{len(df_ciclovias):,}</p></div>
        <div class="kpi"><h3>Estações (ETL4)</h3><p>{len(df_estacoes):,}</p></div>
        <div class="kpi"><h3>Equipamentos (ETL5)</h3><p>{len(df_equip):,}</p></div>
    </div>

    <div class="warning">
        <strong>⚠️ Aviso Analítico:</strong> Dados cadastrais ≠ dados operacionais. Associação descritiva, não causal.
    </div>

    <div class="tabs">
        <button class="tab active" onclick="showTab('tab1')">📞 Chamados</button>
        <button class="tab" onclick="showTab('tab2')">🚨 Infrações</button>
        <button class="tab" onclick="showTab('tab3')">🚴 Malha Cicloviária</button>
        <button class="tab" onclick="showTab('tab4')">🚲 Estações</button>
        <button class="tab" onclick="showTab('tab5')">📷 Fiscalização</button>
    </div>

    <div id="tab1" class="tab-content active">
        <h2>📞 Chamados de Sinistros</h2>
        <div class="chart-container">{grafico_serie_mensal.to_html(full_html=False, include_plotlyjs=False) if grafico_serie_mensal else '<p>Dados não disponíveis</p>'}</div>
        <div class="chart-container">{grafico_matriz_calor.to_html(full_html=False, include_plotlyjs=False) if grafico_matriz_calor else ''}</div>
        <div class="chart-container">{grafico_top_bairros_chamados.to_html(full_html=False, include_plotlyjs=False) if grafico_top_bairros_chamados else ''}</div>
    </div>

    <div id="tab2" class="tab-content">
        <h2>🚨 Infrações de Trânsito</h2>
        <div class="chart-container">{grafico_top_infracoes.to_html(full_html=False, include_plotlyjs=False) if grafico_top_infracoes else '<p>Dados não disponíveis</p>'}</div>
    </div>

    <div id="tab3" class="tab-content">
        <h2>🚴 Malha Cicloviária</h2>
        <div class="chart-container">{grafico_pie_categoria.to_html(full_html=False, include_plotlyjs=False) if grafico_pie_categoria else ''}</div>
        <div class="chart-container">{grafico_densidade.to_html(full_html=False, include_plotlyjs=False) if grafico_densidade else ''}</div>
    </div>

    <div id="tab4" class="tab-content">
        <h2>🚲 Estações de Bicicletas</h2>
        <div class="chart-container">{grafico_estacoes_bairro.to_html(full_html=False, include_plotlyjs=False) if grafico_estacoes_bairro else ''}</div>
        <div class="chart-container">{grafico_proximidade.to_html(full_html=False, include_plotlyjs=False) if grafico_proximidade else ''}</div>
    </div>

    <div id="tab5" class="tab-content">
        <h2>📷 Equipamentos de Fiscalização</h2>
        <div class="chart-container">{grafico_equip_bairro.to_html(full_html=False, include_plotlyjs=False) if grafico_equip_bairro else ''}</div>
        <div class="chart-container">{grafico_dispersao.to_html(full_html=False, include_plotlyjs=False) if grafico_dispersao else ''}</div>
    </div>

    <div class="footer">
        <p><strong>Dashboard Integrado — Mobilidade Urbana Recife 2024</strong></p>
        <p>Desenvolvido com Python, Plotly e SQLite | Dados: Portal Recife Dados</p>
    </div>

    <script>
        function showTab(tabId) {{
            document.querySelectorAll('.tab-content').forEach(el => el.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(el => el.classList.remove('active'));
            document.getElementById(tabId).classList.add('active');
            event.target.classList.add('active');
        }}
    </script>
</body>
</html>
"""

# 4. Salvar HTML
caminho_html = "/content/drive/MyDrive/Backup_ETL5_Recife/dashboard_mobilidade_recife.html"
with open(caminho_html, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"\n🎉 DASHBOARD 100% CORRIGIDO!")
print(f"📁 Local: {caminho_html}")
print(f"💡 Baixe o arquivo e abra no navegador. A legenda agora está perfeita!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ETL1: 5279 registros carregados
✅ ETL2: 404975 registros carregados
✅ ETL3: 367 registros carregados
✅ ETL4: 90 registros carregados
✅ ETL5: 87 registros carregados

🎨 Gerando gráficos com legenda corrigida...
✅ Gráficos gerados com legenda corrigida!

🎉 DASHBOARD 100% CORRIGIDO!
📁 Local: /content/drive/MyDrive/Backup_ETL5_Recife/dashboard_mobilidade_recife.html
💡 Baixe o arquivo e abra no navegador. A legenda agora está perfeita!
